# TFM: Análisis de Políticas de Sostenibilidad mediante técnicas de Argumentacion Computacional

## Detección de Argumentos con phi4 en poliGPT API

In [2]:
%pip install langchain pymupdf openai openpyxl pandas numpy openpyxl --quiet

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
from typing import List
from pydantic import BaseModel, Field, ValidationError
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import PromptTemplate
from langchain_core.exceptions import OutputParserException
import requests
import json
import re
from openai import OpenAI
import openai
import httpx
import pandas as pd
import numpy as np
import os
import openpyxl

process_text_path = "..\\Data\\Processed Files (sections)\\"

model_name="phi4"
prefix = 'GLOBAL_SGD2023'
output_dir = "..\\Data\\Extracted Arguments No Keywords (all text)\\"

## Input text processing

In [2]:
# 1. Define your Pydantic schema for output
class ArgumentResponse(BaseModel):
    arguments: List[str] = Field(..., description="List of arguments extracted directly from the text.")

# 2. Setup output parser
pydantic_parser = PydanticOutputParser(pydantic_object=ArgumentResponse)

# 3. Extend text with first sentence from the next page
def extend_pages_with_next_sentence(pages):
    def get_first_sentence(text):
        match = re.search(r'(.+?\.)', text.strip())
        return match.group(1).strip() if match else ""

    extended_pages = []
    for i, page in enumerate(pages):
        current_text = page["text"]
        if i + 1 < len(pages):
            next_sentence = get_first_sentence(pages[i + 1]["text"])
            current_text += " " + next_sentence
        extended_pages.append({
            "page": page["page"],
            "text": current_text
        })
    return extended_pages

# 4. Build the prompt and call the LLM to extract arguments
def extract_arguments_json(text, topic, model_name) -> ArgumentResponse:
    format_instructions = pydantic_parser.get_format_instructions()

    prompt = PromptTemplate(
        template=(
            "Task: Text Span Identification for Arguments related to Sustainable Development Goal: {topic}\n\n"
            "Role: You are an expert in logical reasoning, sustainability reporting, and argument analysis. "
            "Your job is to identify and extract **verbatim arguments** about {topic} from long-form sustainability texts.\n\n"
            "Instructions:\n"
            "1. Carefully read the entire input text.\n"
            "2. Identify ONLY those sentences or phrases that:\n"
            "   - Clearly support or argue for or against the topic {topic}\n"
            "   - Contain keyword from the relevant lists below\n"
            "   - Are exclusively about {topic} (EXCLUDE if they mention or refer to other SDGs or unrelated sustainability topics)\n\n"
            "3. Each extracted argument must:\n"
            "   - Relate exclusively to the specified SDG ({topic})\n"
            "   - Stand as a full statement\n"
            "   - Be copied exactly from the original (no paraphrasing)\n"
            "   - Include only the necessary context for understanding\n"
            "4. If no qualifying arguments are found, return an empty array.\n\n"
            "Output Rules:\n"
            "- Use **only the exact text** from the original\n"
            "- Do **not** add or reword anything\n"
            "- Return only valid JSON\n"
            "- No markdown (```), no extra explanation\n\n"
            "Text:\n\"\"\"\n{text}\n\"\"\"\n\n"
            "Respond ONLY with a JSON object like this:\n\n"
            "{format_instructions}"
        ),
        input_variables=["text", "topic"],
        partial_variables={"format_instructions": format_instructions}
    )

    final_prompt = prompt.format_prompt(text=text, topic=topic).to_string()

    client = OpenAI(
    base_url = 'https://api.poligpt.upv.es',  
    api_key = 'sk-Icbf-5FyeV0QcLWBC9SNEA'     
        )

    chat_completion = client.chat.completions.create(
        messages = [
            {'role': 'system', 'content': 'You are an expert in logical reasoning, sustainability reporting, and argument analysis.'},
            {'role': 'user', 'content': final_prompt}
        ],
        model = model_name,
        temperature = 0,
    )

    raw_output = chat_completion.choices[0].message.content
    print(raw_output)
    
    try:
        return pydantic_parser.parse(raw_output)
    except OutputParserException as err:
        print("Parse failed:", err)
        return ArgumentResponse(arguments=[])

# 5. Wrapper function for pipeline
def extract_arguments_from_text(text, topic, model_name) -> List[str]:
    result = extract_arguments_json(text, topic, model_name)
    return result.arguments

# 6. Main document-level processor
def process_document(pages, model_name, topic=""):
    extended_pages = extend_pages_with_next_sentence(pages)
    processed = []
    for page in extended_pages:
        print(f"\n--- Processing Page {page['page']} ---")
        #print("Text to analyze:\n", page["text"])
        
        arguments = extract_arguments_from_text(page["text"], topic, model_name)
        
        print("Extracted Arguments:")
        for i, arg in enumerate(arguments, 1):
            print(f"{i}. {arg}")

        processed.append({
            "page": page["page"],
            "text": page["text"],
            "arguments": arguments
        })
    return processed


# 7. File I/O
def save_to_json(processed, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(processed, f, indent=2, ensure_ascii=False)

def process_directory(input_dir, output_dir, prefix, model_name, topic="", sgd_number=None):
    os.makedirs(output_dir, exist_ok=True)
    all_results = []

    for filename in os.listdir(input_dir):
        if filename.endswith(".json") and filename.startswith(prefix):
            filepath = os.path.join(input_dir, filename)
            with open(filepath, "r", encoding="utf-8") as f:
                pages = json.load(f)

            section_name = filename.replace(".json", "")
            processed = process_document(pages, model_name, topic)

            for item in processed:
                item["section"] = section_name  # Add section identifier
                all_results.append(item)
                
    return all_results



## SGD 1: Poverty

In [3]:
topic = "SGD 1 (Poverty): End poverty in all its forms everywhere"
sgd_number = "1"
resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 2: Hunger

In [4]:
topic = "SGD 2 (Hunger): End hunger, achieve food security and improved nutrition and promote sustainable agriculture"
sgd_number = "2"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 3: Health

In [5]:
topic = "SGD 3 (Health): Ensure healthy lives and promote well-being for all at all ages"
sgd_number = "3"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 4: Education

In [6]:
topic = "SGD 4 (Education): Ensure inclusive and equitable quality education"
sgd_number = "4"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 5: Gender

In [7]:
topic = "SGD 5 (Gender): Achieve gender equality and empower all women and girls"
sgd_number = "5"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 6: Water and sanitation

In [8]:
topic = "SGD 6 (Water and sanitation): Ensure availability and sustainable management of water and sanitation for all"
sgd_number = "6"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)




--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 7: Clean Energy

In [9]:
topic = "SGD 7 (Clean Energy): Ensure access to affordable, reliable, sustainable and modern energy for all"
sgd_number = "7"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 8: Decent Work, Economic Growth

In [10]:
topic = "SGD 8 (decent work, economic growth): Promote sustained, inclusive and sustainable economic growth, full and productive employment and decent work for all"
sgd_number = "8"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 9: Infrastructure, industrilization, innovation

In [11]:
topic = "SGD 9 (Infrastructure, industrilization, innovation): Build resilient infrastructure, promote inclusive and sustainable industrialization and foster innovation"
sgd_number = "9"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 10: Inequality

In [12]:
topic = "SGD 10 (Inequality): Reduce inequality within and among countries"
sgd_number = "10"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 11: Sustainable cities

In [13]:
topic = "SGD 11 (Sustainable Cities, Sustainable Communities): Make cities and human settlements inclusive, safe, resilient and sustainable"
sgd_number = "11"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]

Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to 

## SGD 12: Responsible Consumption, Responsible Production

In [14]:
topic = "SGD 12 (Responsible Consumption, Responsible Production): Ensure sustainable consumption and production patterns"
sgd_number = "12"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 13: Climate change

In [15]:
topic = "SGD 13 (Climate change): Take urgent action to combat climate change and its impacts"
sgd_number = "13"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 14: Life bellow water

In [16]:
topic = "SGD 14 (Life bellow Water): Conserve and sustainably use the oceans, seas and marine resources for sustainable development"
sgd_number = "14"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 15: Life on land

In [17]:
topic = "SGD 15 (Life on land): Protect, restore and promote sustainable use of terrestrial ecosystems, sustainably manage forests, combat desertification, and halt and reverse land degradation and halt biodiversity loss"
sgd_number = "15"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 16: Peace, Justice, Strong Institutions

In [18]:
topic = "SGD 16 (Peace, Justice, Strong Institutions): Promote peaceful and inclusive societies for sustainable development, provide access to justice for all and build effective, accountable and inclusive institutions at all levels"
sgd_number = "16"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 17: Partnerships, sustainable development

In [19]:
topic = "SGD 17 (Partnerships, sustainable development):Strengthen the means of implementation and revitalize the Global Partnership for Sustainable Development"
sgd_number = "17"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p

## SGD 0: Overarching terms

In [20]:
topic = "SGD Overarching terms: Sustainable Development Goal, SDG, Agenda 2030, leave no one behind, Voluntary National Review, SDG transformations, "
sgd_number = "0"

resultado = process_directory(input_dir = process_text_path, 
                  output_dir = output_dir, 
                  prefix = prefix,
                  model_name = model_name,
                  topic = topic,
                  sgd_number =sgd_number)

merged_output_path = os.path.join(output_dir, f"{prefix}_ArgsSGD{sgd_number}_{model_name.replace(':', '-')}.json")
save_to_json(resultado, merged_output_path)


--- Processing Page 7 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 8 ---
[]
Parse failed: Failed to parse ArgumentResponse from completion []. Got: 1 validation error for ArgumentResponse
  Input should be a valid dictionary or instance of ArgumentResponse [type=model_type, input_value=[], input_type=list]
    For further information visit https://errors.pydantic.dev/2.11/v/model_type
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 
Extracted Arguments:

--- Processing Page 13 ---
[]
Parse failed: Failed to p